In [1]:
# ============================================================
# Startup Cell: Mount Drive + Load Config + Verify Inputs
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import sys

# ------------------------------------------------------------
# Add project source directory to Python path
# ------------------------------------------------------------
SRC_DIR = "/content/drive/MyDrive/DIP_Project/src"

if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)

# ------------------------------------------------------------
# Import project configuration
# ------------------------------------------------------------
from project_config import *

print("Project configuration loaded.")
print(f"BASE_DIR:     {BASE_DIR}")
print(f"METADATA_DIR: {METADATA_DIR}")
print(f"MODELS_DIR:   {MODELS_DIR}")

# ------------------------------------------------------------
# Verify required input files
# ------------------------------------------------------------
print("\nVerifying required normalized feature files...")

required_files = [
    os.path.join(METADATA_DIR, TRAIN_NORMALIZED_FILENAME),
    os.path.join(METADATA_DIR, TEST_NORMALIZED_FILENAME),
]

all_files_present = True

for file_path in required_files:
    if os.path.exists(file_path):
        print(f"{os.path.basename(file_path):50s} OK")
    else:
        print(f"{os.path.basename(file_path):50s} MISSING")
        all_files_present = False

# ------------------------------------------------------------
# Verify / create output directories
# ------------------------------------------------------------
print("\nVerifying output directories...")

required_dirs = [
    MODELS_DIR,
]

for dir_path in required_dirs:
    if not os.path.exists(dir_path):
        os.makedirs(dir_path, exist_ok=True)
        print(f"Created: {dir_path}")
    else:
        print(f"Exists:  {dir_path}")

# ------------------------------------------------------------
# Display key runtime parameters (sanity check)
# ------------------------------------------------------------
print("\nKey configuration values:")
print(f"Number of features: {NUM_FEATURES}")
print(f"Train/Test split:   {TRAIN_RATIO:.2f} / {TEST_RATIO:.2f}")
print(f"K-Folds (CV):       {K_FOLDS}")
print(f"Random seed:        {RANDOM_SEED}")

# ------------------------------------------------------------
# Final status
# ------------------------------------------------------------
if all_files_present:
    print("\nAll required files are present. Ready to proceed.")
else:
    print("\nERROR: Missing required input files. Check paths before continuing.")



Mounted at /content/drive
Project configuration loaded.
BASE_DIR:     /content/drive/MyDrive/DIP_Project
METADATA_DIR: /content/drive/MyDrive/DIP_Project/data/metadata
MODELS_DIR:   /content/drive/MyDrive/DIP_Project/models

Verifying required normalized feature files...
train_feature_vectors_normalized.csv               OK
test_feature_vectors_normalized.csv                OK

Verifying output directories...
Exists:  /content/drive/MyDrive/DIP_Project/models

Key configuration values:
Number of features: 25
Train/Test split:   0.80 / 0.20
K-Folds (CV):       5
Random seed:        42

All required files are present. Ready to proceed.


In [2]:
# ============================================================
# Cell 0: Notebook Overview
# ============================================================
#
# Notebook:
# 08_Train_Neural_Network.ipynb
#
# Purpose:
# Train and compare a small set of candidate Radial Basis
# Function Support Vector Machine (RBF SVM) classifiers using
# the normalized Digital Image Processing (DIP) feature
# vectors generated in earlier pipeline steps.
#
# This notebook provides a transparent follow-on to Notebook 07
# by exposing how different RBF SVM hyperparameter settings
# affect cross-validation performance on the training set.
# After comparing candidate configurations, the best-performing
# model is trained on the full training dataset and saved for
# final independent evaluation in Notebook 09.
#
# ------------------------------------------------------------
# Inputs:
# ------------------------------------------------------------
# Normalized training and test datasets produced in
# Notebook 06:
#
#   - X_train
#   - y_train
#   - X_test
#   - y_test
#
# These datasets:
#   - contain the full DIP feature vectors (25 features)
#   - are normalized using training-set statistics
#   - are free of missing values
#
# ------------------------------------------------------------
# Assumptions:
# ------------------------------------------------------------
# - Feature extraction and normalization have been completed
# - Broad classifier selection has already been performed in
#   Notebook 07
# - The dataset is split into training and test sets only
# - Cross-validation is applied only to the training set
# - The test set remains unused in this notebook
#
# ------------------------------------------------------------
# Cell-by-Cell Flow:
# ------------------------------------------------------------
# Cell 1:
#   Environment setup and imports
#
# Cell 2:
#   Load normalized training and test data
#
# Cell 3:
#   Run sanity checks on loaded data and separate features
#   and labels
#
# Cell 4:
#   Define candidate RBF SVM configurations using selected
#   values of C and gamma
#
# Cell 5:
#   Evaluate each candidate configuration using stratified
#   cross-validation on the training set
#
# Cell 6:
#   Summarize and rank cross-validation results in tabular
#   form
#
# Cell 7:
#   Define and train the best-performing RBF SVM model on
#   the full training dataset
#
# Cell 8:
#   Save the trained final model to disk
#
# Cell 9:
#   Save the cross-validation summary and best-model
#   configuration for Notebook 09
#
# ------------------------------------------------------------
# Outputs:
# ------------------------------------------------------------
# - Cross-validation comparison of candidate RBF SVM models
# - Ranked model summary table
# - Best RBF SVM configuration
# - Trained final model saved to disk
# - Saved CSV and JSON outputs for downstream evaluation
#
# ------------------------------------------------------------
# Notes:
# ------------------------------------------------------------
# - This notebook does NOT evaluate performance on the test set
# - Final independent test evaluation is performed in Notebook 09
# - This notebook makes the selected classifier family more
#   transparent by exposing a small RBF SVM parameter study
#
# ============================================================

print("Notebook overview loaded.")



Notebook overview loaded.


In [3]:
# ============================================================
# Cell 1: Imports
# ============================================================

import os
import json
import pickle
import warnings

import numpy as np
import pandas as pd

import time

from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

warnings.filterwarnings("ignore")

print("Imports loaded successfully.")


Imports loaded successfully.


In [4]:
# ============================================================
# Cell 2: Load Normalized Training and Test Data
# ============================================================

train_csv_path = os.path.join(METADATA_DIR, TRAIN_NORMALIZED_FILENAME)
test_csv_path = os.path.join(METADATA_DIR, TEST_NORMALIZED_FILENAME)

df_train = pd.read_csv(train_csv_path)
df_test = pd.read_csv(test_csv_path)

print("Normalized feature tables loaded successfully.\n")

print(f"Training CSV: {train_csv_path}")
print(f"Test CSV:     {test_csv_path}\n")

print(f"df_train shape: {df_train.shape}")
print(f"df_test shape:  {df_test.shape}\n")

print("First 5 rows of training table:")
display(df_train.head())



Normalized feature tables loaded successfully.

Training CSV: /content/drive/MyDrive/DIP_Project/data/metadata/train_feature_vectors_normalized.csv
Test CSV:     /content/drive/MyDrive/DIP_Project/data/metadata/test_feature_vectors_normalized.csv

df_train shape: (14400, 29)
df_test shape:  (3600, 29)

First 5 rows of training table:


,filename,class_label,source_dataset,subset,Mean Gradient,Std Gradient,Max Gradient,Gradient Entropy,Edge Density,Orientation Mean,...,Patch Variance Std,Noise Residual Energy,Low Frequency Energy Ratio,High Frequency Energy Ratio,Radial Mean,Radial Std,Radial Entropy,Spectral Centroid,Spectral Bandwidth,Log Spectrum Std
0,rl_imgn_002320.png,rl,ImageNet_1K_256,train,0.973214,1.208264,0.913526,0.776979,0.735101,-0.207859,...,1.019527,1.071297,-0.189219,0.417247,0.448644,0.439427,-0.549996,-0.084929,0.290945,-1.254194
1,rl_coco_001397.png,rl,MS_COCO_2017,train,-0.450913,-0.225225,0.641767,-0.488332,-0.447341,-0.246587,...,0.509586,-0.330229,-0.264290,0.144170,-1.327122,-1.325683,1.397134,0.856623,0.808074,0.071118
2,rl_imgn_001958.png,rl,ImageNet_1K_256,train,0.605933,0.723094,0.717432,0.610900,0.160581,-0.468100,...,0.235488,0.851489,-0.544033,0.805334,-0.343146,-0.332461,-0.549996,-0.217414,0.432281,-1.341491
3,rl_coco_000800.png,rl,MS_COCO_2017,train,0.021388,0.357107,0.528482,-0.077406,0.331237,1.075664,...,0.395378,0.188882,0.518813,-0.349898,1.963337,1.956027,-0.549996,-0.448685,-0.628062,-1.111831
4,ai_mj_002892.png,ai,Midjourney,train,-0.212459,-0.661686,-0.209270,0.288772,-0.061089,-0.293840,...,-0.373574,-0.281104,0.607047,-0.556607,-0.011009,-0.013482,-0.549996,-0.185376,-0.470299,0.325939


In [5]:
# ============================================================
# Cell 3: Sanity Checks + Separate Features and Labels
# ============================================================

# Expected metadata columns
expected_metadata_columns = METADATA_COLUMNS.copy()

# Verify required metadata columns
missing_train_metadata = [col for col in expected_metadata_columns if col not in df_train.columns]
missing_test_metadata = [col for col in expected_metadata_columns if col not in df_test.columns]

if missing_train_metadata:
    raise ValueError(f"Training table is missing metadata columns: {missing_train_metadata}")

if missing_test_metadata:
    raise ValueError(f"Test table is missing metadata columns: {missing_test_metadata}")

print("Required metadata columns verified.")

# Verify feature count
feature_columns = [col for col in df_train.columns if col not in METADATA_COLUMNS]

if len(feature_columns) != NUM_FEATURES:
    raise ValueError(
        f"Expected {NUM_FEATURES} feature columns, but found {len(feature_columns)}."
    )

print(f"Feature count verified: {len(feature_columns)}")

# Verify test table has same feature columns in same order
test_feature_columns = [col for col in df_test.columns if col not in METADATA_COLUMNS]

if feature_columns != test_feature_columns:
    raise ValueError("Training and test feature columns do not match exactly.")

print("Training and test feature columns match.")

# Verify no missing values
if df_train.isnull().sum().sum() != 0:
    raise ValueError("Training table contains missing values.")

if df_test.isnull().sum().sum() != 0:
    raise ValueError("Test table contains missing values.")

print("No missing values detected.")

# Separate features and labels
X_train = df_train[feature_columns].copy()
X_test = df_test[feature_columns].copy()

y_train = df_train["class_label"].copy()
y_test = df_test["class_label"].copy()

# Display class distributions
print("\nTraining class counts:")
print(y_train.value_counts())

print("\nTest class counts:")
print(y_test.value_counts())

# Display feature matrix shapes
print("\nFeature matrix shapes:")
print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")

print("\nLabel vector shapes:")
print(f"y_train: {y_train.shape}")
print(f"y_test:  {y_test.shape}")

print("\nSanity checks passed successfully.")



Required metadata columns verified.
Feature count verified: 25
Training and test feature columns match.
No missing values detected.

Training class counts:
class_label
rl    7200
ai    7200
Name: count, dtype: int64

Test class counts:
class_label
rl    1800
ai    1800
Name: count, dtype: int64

Feature matrix shapes:
X_train: (14400, 25)
X_test:  (3600, 25)

Label vector shapes:
y_train: (14400,)
y_test:  (3600,)

Sanity checks passed successfully.


In [6]:
# ============================================================
# Cell 4: Define Candidate RBF SVM Configurations
# ============================================================

candidate_configs = [
    {"model_name": "RBF_SVM_C1_gamma0.01",   "C": 1.0,   "gamma": 0.01},
    {"model_name": "RBF_SVM_C1_gammascale",  "C": 1.0,   "gamma": "scale"},
    {"model_name": "RBF_SVM_C1_gamma0.1",    "C": 1.0,   "gamma": 0.1},
    {"model_name": "RBF_SVM_C10_gamma0.01",  "C": 10.0,  "gamma": 0.01},
    {"model_name": "RBF_SVM_C10_gammascale", "C": 10.0,  "gamma": "scale"},
    {"model_name": "RBF_SVM_C10_gamma0.1",   "C": 10.0,  "gamma": 0.1},
    {"model_name": "RBF_SVM_C100_gamma0.01", "C": 100.0, "gamma": 0.01},
    {"model_name": "RBF_SVM_C100_gammascale","C": 100.0, "gamma": "scale"},
    {"model_name": "RBF_SVM_C100_gamma0.1",  "C": 100.0, "gamma": 0.1},
]

print("Candidate RBF SVM configurations defined successfully.\n")
print(f"Number of candidate models: {len(candidate_configs)}\n")

for config in candidate_configs:
    print(
        f"{config['model_name']}: "
        f"C={config['C']}, gamma={config['gamma']}"
    )



Candidate RBF SVM configurations defined successfully.

Number of candidate models: 9

RBF_SVM_C1_gamma0.01: C=1.0, gamma=0.01
RBF_SVM_C1_gammascale: C=1.0, gamma=scale
RBF_SVM_C1_gamma0.1: C=1.0, gamma=0.1
RBF_SVM_C10_gamma0.01: C=10.0, gamma=0.01
RBF_SVM_C10_gammascale: C=10.0, gamma=scale
RBF_SVM_C10_gamma0.1: C=10.0, gamma=0.1
RBF_SVM_C100_gamma0.01: C=100.0, gamma=0.01
RBF_SVM_C100_gammascale: C=100.0, gamma=scale
RBF_SVM_C100_gamma0.1: C=100.0, gamma=0.1


In [7]:
# ============================================================
# Cell 5: Cross-Validate Candidate RBF SVM Models
# ============================================================

import time

cv = StratifiedKFold(
    n_splits=5,
    shuffle=CV_SHUFFLE,
    random_state=CV_RANDOM_STATE
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision_macro",
    "recall": "recall_macro",
    "f1": "f1_macro",
    "roc_auc": "roc_auc",
}

results = []

print("Evaluating candidate RBF SVM models...\n")
print(f"Number of models: {len(candidate_configs)}")
print("Cross-validation folds: 5\n")

total_start_time = time.time()

for i, config in enumerate(candidate_configs, start=1):
    model_start_time = time.time()

    print(
        f"[{i}/{len(candidate_configs)}] "
        f"{config['model_name']} "
        f"(C={config['C']}, gamma={config['gamma']})"
    )

    model = SVC(
        C=config["C"],
        gamma=config["gamma"],
        kernel="rbf",
        probability=True,
        random_state=RANDOM_SEED
    )

    cv_results = cross_validate(
        estimator=model,
        X=X_train,
        y=y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=False,
        n_jobs=-1
    )

    row = {
        "model_name": config["model_name"],
        "C": config["C"],
        "gamma": config["gamma"],
        "accuracy_mean": cv_results["test_accuracy"].mean(),
        "accuracy_std": cv_results["test_accuracy"].std(),
        "precision_mean": cv_results["test_precision"].mean(),
        "precision_std": cv_results["test_precision"].std(),
        "recall_mean": cv_results["test_recall"].mean(),
        "recall_std": cv_results["test_recall"].std(),
        "f1_mean": cv_results["test_f1"].mean(),
        "f1_std": cv_results["test_f1"].std(),
        "roc_auc_mean": cv_results["test_roc_auc"].mean(),
        "roc_auc_std": cv_results["test_roc_auc"].std(),
    }

    results.append(row)

    model_elapsed = time.time() - model_start_time
    total_elapsed = time.time() - total_start_time

    print(f"  Completed in {model_elapsed:.2f} seconds")
    print(f"  Total elapsed: {total_elapsed:.2f} seconds\n")

print("Cross-validation complete.")
print(f"Total runtime: {time.time() - total_start_time:.2f} seconds")



Evaluating candidate RBF SVM models...

Number of models: 9
Cross-validation folds: 5

[1/9] RBF_SVM_C1_gamma0.01 (C=1.0, gamma=0.01)
  Completed in 33.71 seconds
  Total elapsed: 33.71 seconds

[2/9] RBF_SVM_C1_gammascale (C=1.0, gamma=scale)
  Completed in 31.27 seconds
  Total elapsed: 64.98 seconds

[3/9] RBF_SVM_C1_gamma0.1 (C=1.0, gamma=0.1)
  Completed in 31.88 seconds
  Total elapsed: 96.85 seconds

[4/9] RBF_SVM_C10_gamma0.01 (C=10.0, gamma=0.01)
  Completed in 30.92 seconds
  Total elapsed: 127.77 seconds

[5/9] RBF_SVM_C10_gammascale (C=10.0, gamma=scale)
  Completed in 32.20 seconds
  Total elapsed: 159.97 seconds

[6/9] RBF_SVM_C10_gamma0.1 (C=10.0, gamma=0.1)
  Completed in 37.03 seconds
  Total elapsed: 197.01 seconds

[7/9] RBF_SVM_C100_gamma0.01 (C=100.0, gamma=0.01)
  Completed in 40.34 seconds
  Total elapsed: 237.35 seconds

[8/9] RBF_SVM_C100_gammascale (C=100.0, gamma=scale)
  Completed in 57.26 seconds
  Total elapsed: 294.61 seconds

[9/9] RBF_SVM_C100_gamma0.1 

In [8]:
# ============================================================
# Cell 6: Summarize and Rank Candidate Model Results
# ============================================================

df_cv_results = pd.DataFrame(results)
df_cv_results = df_cv_results.sort_values(
    by="roc_auc_mean",
    ascending=False
).reset_index(drop=True)

print("Candidate model comparison (sorted by ROC-AUC):\n")
display(df_cv_results)

best_row = df_cv_results.iloc[0]

best_model_name = best_row["model_name"]
best_params = {
    "C": float(best_row["C"]),
    "gamma": best_row["gamma"],
    "kernel": "rbf",
    "probability": True,
    "random_state": RANDOM_SEED,
}

print("Best configuration identified:\n")
print(f"Model: {best_model_name}")
for key, value in best_params.items():
    print(f"  {key}: {value}")


Candidate model comparison (sorted by ROC-AUC):



,model_name,C,gamma,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std
0,RBF_SVM_C100_gamma0.01,100.0,0.01,0.798194,0.008415,0.798329,0.008319,0.798194,0.008415,0.798171,0.008434,0.879229,0.005399
1,RBF_SVM_C10_gammascale,10.0,scale,0.799097,0.008416,0.799221,0.008331,0.799097,0.008416,0.799075,0.008432,0.878957,0.005807
2,RBF_SVM_C100_gammascale,100.0,scale,0.795694,0.007775,0.795795,0.007684,0.795694,0.007775,0.795676,0.007793,0.874585,0.006369
3,RBF_SVM_C10_gamma0.1,10.0,0.1,0.787708,0.004599,0.787759,0.004607,0.787708,0.004599,0.787699,0.004598,0.866250,0.003696
4,RBF_SVM_C10_gamma0.01,10.0,0.01,0.782847,0.008692,0.782952,0.008613,0.782847,0.008692,0.782826,0.008709,0.865653,0.006044
5,RBF_SVM_C1_gamma0.1,1.0,0.1,0.783403,0.002824,0.783456,0.002838,0.783403,0.002824,0.783393,0.002822,0.862251,0.004222
6,RBF_SVM_C1_gammascale,1.0,scale,0.777778,0.005835,0.777879,0.005844,0.777778,0.005835,0.777758,0.005834,0.861125,0.005117
7,RBF_SVM_C1_gamma0.01,1.0,0.01,0.757361,0.004175,0.757538,0.004078,0.757361,0.004175,0.757319,0.004200,0.839984,0.006813
8,RBF_SVM_C100_gamma0.1,100.0,0.1,0.764444,0.006531,0.764590,0.006427,0.764444,0.006531,0.764411,0.006557,0.838952,0.003898


Best configuration identified:

Model: RBF_SVM_C100_gamma0.01
  C: 100.0
  gamma: 0.01
  kernel: rbf
  probability: True
  random_state: 42


In [9]:
# ============================================================
# Cell 7: Train Best RBF SVM Model on Full Training Set
# ============================================================

final_model = SVC(**best_params)
final_model.fit(X_train, y_train)

print("Final model trained successfully.\n")
print(f"Model: {best_model_name}")



Final model trained successfully.

Model: RBF_SVM_C100_gamma0.01


In [10]:
# ============================================================
# Cell 8: Save Trained Final Model
# ============================================================

final_model_path = os.path.join(MODELS_DIR, "final_rbf_svm_model.pkl")

with open(final_model_path, "wb") as f:
    pickle.dump(final_model, f)

print("Final model saved successfully.")
print(f"Saved to: {final_model_path}")



Final model saved successfully.
Saved to: /content/drive/MyDrive/DIP_Project/models/final_rbf_svm_model.pkl


In [11]:
# ============================================================
# Cell 9: Save Cross-Validation Summary and Best Model Config
# ============================================================

cv_results_path = os.path.join(METADATA_DIR, CV_RESULTS_FILENAME)
best_config_path = os.path.join(MODELS_DIR, BEST_MODEL_CONFIG_FILENAME)

# Save cross-validation summary table
df_cv_results.to_csv(cv_results_path, index=False)

# Save best model configuration
best_model_config = {
    "model_name": best_model_name,
    "model_type": "RBF SVM",
    "hyperparameters": best_params,
    "num_features": NUM_FEATURES,
    "k_folds": K_FOLDS,
    "train_samples": int(len(X_train)),
    "test_samples": int(len(X_test)),
}

with open(best_config_path, "w") as f:
    json.dump(best_model_config, f, indent=4)

print("Cross-validation summary saved successfully.")
print(f"CSV:  {cv_results_path}")

print("\nBest model configuration saved successfully.")
print(f"JSON: {best_config_path}")



Cross-validation summary saved successfully.
CSV:  /content/drive/MyDrive/DIP_Project/data/metadata/cross_validation_results.csv

Best model configuration saved successfully.
JSON: /content/drive/MyDrive/DIP_Project/models/best_model_config.json
